[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shripada/ame5003-nlp/blob/main/labs/lab-09-rnn-lstm.ipynb)

**Click the badge above to open this lab in Google Colab.** Then choose *File → Save a copy in Drive* so your work is saved.

# Lab 9 — Sequence models: the RNN and the LSTM

**MSIS · AME 5053 · Week 9 · 3 hours**

Three labs have now turned a review into numbers, and every one of them threw away word order.
Lab 6 counted words, lab 7 weighted those counts by tf-idf, and lab 8 averaged word vectors —
and session 23 showed what averaging costs, exactly: the cosine between *dog bites man* and
*man bites dog*, built from averaged GloVe vectors, is 0.9999999999999998. The two sentences
are the same point in space.

Session 23 also gave the architecture that does not have this problem. An RNN reads a document
one word at a time, carrying a hidden state forward, so <code>h_t</code> depends on everything
before it *in order*. This lab builds one in PyTorch, then builds the LSTM session 25 motivated,
and measures both on the task labs 6, 7 and 8 have been measuring all along.

**By the end of this lab you will be able to:**

1. Turn variable-length documents into padded tensors, with a vocabulary built from the training split
2. Build a sequence classifier around `nn.RNN` and read its output shapes with confidence
3. Explain why a padded final hidden state is the wrong vector, and fix it with `pack_padded_sequence`
4. Swap in `nn.LSTM` and `nn.GRU`, and say what changed in the API and what changed in the result
5. Compare a sequence model against tf-idf honestly, and report the answer the measurement gives

---

## Part 0 — Setup

Same corpus and the same split as labs 6, 7 and 8, so every number in this lab can be put beside
the numbers those labs produced. PyTorch is already installed on Colab; the corpus download is a
fresh network fetch every session.

`gensim` is **not** preinstalled on Colab and is installed below — Part 2 needs it for the GloVe
vectors. If Colab reports a dependency conflict and offers to restart the session, accept, then run
this cell again before continuing.

In [ ]:
%pip install -q nltk scikit-learn gensim

import nltk
ok = nltk.download("movie_reviews")
print("movie_reviews downloaded:", ok)

import numpy as np
import torch
import torch.nn as nn
print("torch", torch.__version__, "· threads", torch.get_num_threads())
print("Done.")

### Where this will run

The lab is written to work with or without a GPU. If Colab has given you one
(Runtime → Change runtime type → T4 GPU) the models train there; if not, everything runs on the
CPU and the only difference is how long you wait.

Ask for a GPU if you can get one, but do not depend on it — Google does not promise one, and this
lab was written and its numbers measured on CPU.

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("training on:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

> **Save your own copy now:** File → Save a copy in Drive.

The split is labs 6–8's split, seed and all: 1,500 documents to train on, 500 held out.

In [ ]:
from nltk.corpus import movie_reviews
from sklearn.model_selection import train_test_split

ids = movie_reviews.fileids()
texts = [movie_reviews.raw(i) for i in ids]
labels = np.array([1 if i.startswith("pos") else 0 for i in ids])

Xtr_text, Xte_text, ytr, yte = train_test_split(
    texts, labels, test_size=500, random_state=42, stratify=labels)

print("train:", len(Xtr_text), "  test:", len(Xte_text))

---

## Part 1 — From documents to tensors

A network cannot take a list of strings. Every document has to become a row of integers, all rows
the same width, and that requirement is the whole of this part. It is also the part with no
counterpart in labs 6, 7 or 8, where scikit-learn did this for us.

We reuse lab 8's tokenizer unchanged.

In [ ]:
import re

token_re = re.compile(r"[a-z]+")

def tokenize(text):
    """Lowercase, keep runs of letters only."""
    return token_re.findall(text.lower())

tokenized_train = [tokenize(t) for t in Xtr_text]
tokenized_test  = [tokenize(t) for t in Xte_text]

lengths = np.array([len(t) for t in tokenized_train])
print("mean length  :", round(lengths.mean()))
print("median       :", int(np.median(lengths)))
print("90th pct     :", int(np.percentile(lengths, 90)))
print("longest       :", lengths.max())

# Verified output:
#   mean length  : 662
#   median       : 623
#   90th pct     : 1017
#   longest      : 2459

### Which 300 words?

We will cut every document to 300 tokens. That is not a small cut — 93% of these reviews are
longer than 300 tokens, so for almost every document we are choosing which part to keep.

The obvious choice is the first 300, and it is wrong. A film review spends its opening on plot
summary and arrives at its verdict at the end. Holding everything else fixed, the same LSTM
scores **0.552** on the first 300 tokens and **0.776** on the last 300. Nothing in the code
fails when you choose wrongly; the accuracy simply collapses.

We keep the **last** 300.

In [ ]:
from collections import Counter

def build_vocab(tokenized_docs, min_count=2, max_size=20000):
    """Map word -> index. Index 0 is <pad>, index 1 is <unk>.

    Only words appearing at least min_count times in the TRAINING documents get an index.
    """
    # YOUR CODE HERE
    # 1. Counter over every token in every document
    # 2. keep words with count >= min_count, most common first, at most max_size of them
    # 3. itos = ["<pad>", "<unk>"] + those words
    # 4. return {word: index}, itos
    pass

In [ ]:
stoi, itos = build_vocab(tokenized_train)
print("vocabulary size:", len(itos))
print("first ten words:", itos[:10])

# Verified output:
#   vocabulary size: 20002
#   first ten words: ['<pad>', '<unk>', 'the', 'a', 'and', 'of', 'to', 'is', 'in', 's']
#
# The tenth entry is 's', not a word — our tokenizer keeps runs of letters only, so every
# possessive and contraction in the corpus ("movie's", "it's") leaves an 's' behind. Lab 3
# met the same thing. It is frequent enough to make the top ten.

### Padding, and the length that goes with it

Every row must be the same width, so short documents are padded with index 0. The padding is not
information, and the model must never be allowed to treat it as information — which is why we
keep the **true length** of every document alongside the padded row. Everything in Part 3 depends
on having kept it.

In [ ]:
MAX_LEN = 300

def encode(tokenized_docs, stoi, max_len=MAX_LEN):
    """Return (padded LongTensor of shape [n_docs, width], LongTensor of true lengths).

    Keep the LAST max_len tokens of each document. Unknown words map to index 1.
    """
    # YOUR CODE HERE
    # 1. ids = [stoi.get(t, 1) for t in doc], then take the last max_len of them
    # 2. lengths = the length of each id list (never zero — fall back to [1])
    # 3. allocate torch.zeros(n_docs, max(lengths), dtype=torch.long) and fill each row
    pass

In [ ]:
Xtr, ltr = encode(tokenized_train, stoi)
Xte, lte = encode(tokenized_test, stoi)

print("training tensor:", tuple(Xtr.shape), "· lengths:", tuple(ltr.shape))
print("shortest document:", ltr.min().item(), "tokens · longest kept:", ltr.max().item())

assert Xtr.shape[0] == 1500 and Xte.shape[0] == 500
assert ltr.max().item() <= MAX_LEN
assert (Xtr[0, ltr[0]:] == 0).all(), "everything past the true length must be padding"
print("Shapes and padding check out.")

---

## Part 2 — The embedding layer, started from GloVe

The first layer turns each index into a vector. It can be trained from scratch, and on a large
corpus that is what one does — but 1,500 documents is not a large corpus, and lab 8 already
downloaded 400,000 pretrained GloVe vectors.

We copy those vectors into the embedding layer as its starting point. Words the vocabulary has
but GloVe does not keep a small random vector. Training continues to adjust all of them.

This is worth about four points of accuracy, which the cell below records. It is a real gain and a
modest one — starting from random vectors still gets to 0.72–0.73 here, so pretrained embeddings
are helping rather than rescuing. Keep the size of that number in mind at the end of the lab,
where a much larger version of the same idea is the subject of session 32.

In [ ]:
import gensim.downloader as api

kv = api.load("glove-wiki-gigaword-100")           # ~128MB the first time
EMB_DIM = 100

emb_init = np.random.RandomState(42).normal(0, 0.1, (len(itos), EMB_DIM)).astype("float32")
hits = 0
for i, w in enumerate(itos):
    if w in kv:
        emb_init[i] = kv[w]
        hits += 1
emb_init[0] = 0.0                                   # <pad> starts at the origin

print(f"GloVe vectors found for {hits} of {len(itos)} vocabulary words "
      f"({hits / len(itos) * 100:.0f}%)")

# Verified output:
#   GloVe vectors found for 19736 of 20002 vocabulary words (99%)
#
# What this is worth, measured (last 300 tokens, mean-pooled, 8 epochs):
#   LSTM, GloVe start : 0.760      LSTM, random start : 0.718
#   RNN,  GloVe start : 0.766      RNN,  random start : 0.734

---

## Part 3 — The Elman RNN, and the padding trap

Session 23's recurrence, in one line of PyTorch:

    self.rnn = nn.RNN(EMB_DIM, HIDDEN, batch_first=True)

`nn.RNN` returns two things: the hidden state at *every* step, and the hidden state at the
**last** step. For classification session 23 used the second one — run the whole document
through, take <code>h_n</code>, feed it to a small linear layer.

There is a trap in that sentence, and it is worth meeting before the model that hides it.

In [ ]:
HIDDEN = 128

demo = nn.RNN(4, 3, batch_first=True)
x = torch.randn(1, 6, 4)                # one sequence, 6 steps, 4 features
x_padded = torch.cat([x, torch.zeros(1, 4, 4)], dim=1)   # same sequence + 4 pad steps

h_true = demo(x)[1]
h_padded = demo(x_padded)[1]

print("final state, 6 real steps      :", h_true.squeeze().detach().numpy().round(4))
print("final state, 6 real + 4 padded :", h_padded.squeeze().detach().numpy().round(4))
print("same?", torch.allclose(h_true, h_padded))

The two are different, and the second one is wrong. `nn.RNN` has no idea that index 0 means
"nothing here" — it applies the recurrence to the padding exactly as it would to a word, so the
final hidden state of a padded batch is the state after four steps of nothing. A short document
in a batch of long ones gets its meaning diluted, and no error is raised.

The fix is `pack_padded_sequence`, which takes the true lengths we kept in Part 1 and tells the
RNN where each sequence genuinely ends.

In [ ]:
class SeqClassifier(nn.Module):
    """Embedding -> recurrent layer -> linear head. cell is 'rnn', 'lstm' or 'gru'."""

    def __init__(self, vocab_size, emb_init=None, cell="rnn", pool="last",
                 emb_dim=EMB_DIM, hidden=HIDDEN, n_classes=2):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        if emb_init is not None:
            self.emb.weight.data.copy_(torch.from_numpy(emb_init))
        self.rnn = {"rnn": nn.RNN, "lstm": nn.LSTM, "gru": nn.GRU}[cell](
            emb_dim, hidden, batch_first=True)
        self.cell, self.pool = cell, pool
        self.head = nn.Linear(hidden, n_classes)

    def forward(self, x, lens):
        e = self.emb(x)
        packed = nn.utils.rnn.pack_padded_sequence(
            e, lens, batch_first=True, enforce_sorted=False)
        out, last = self.rnn(packed)

        if self.pool == "last":
            # nn.LSTM returns (h_n, c_n); nn.RNN and nn.GRU return h_n alone
            h_n = last[0] if self.cell == "lstm" else last
            h = h_n[-1]
        else:
            seq, l = nn.utils.rnn.pad_packed_sequence(out, batch_first=True)
            # pad_packed_sequence hands the lengths back on the CPU, always. On a GPU run,
            # seq is on the GPU and l is not, so both the mask and the divisor have to be
            # moved before they meet it.
            l = l.to(seq.device)
            mask = (torch.arange(seq.size(1), device=seq.device)[None, :] < l[:, None]).unsqueeze(-1)
            h = (seq * mask).sum(1) / l[:, None]     # mean over real steps only

        return self.head(h)

### The training loop

Nothing here is specific to sequences — it is the loop session 20 built, with Adam in place of
plain gradient descent and gradient clipping added, which session 24 explained as the standard
guard against the exploding half of the gradient problem.

In [ ]:
def evaluate(model, X, lens, y, bs=64):
    model.eval()
    preds = []
    with torch.no_grad():
        for i in range(0, len(y), bs):
            # lengths stay on the CPU: pack_padded_sequence requires that, on every device
            out = model(X[i:i + bs].to(DEVICE), lens[i:i + bs])
            preds.append(out.argmax(1).cpu())
    return float((torch.cat(preds).numpy() == y).mean())


def train(cell="rnn", pool="last", emb_init=emb_init, epochs=8, bs=32, lr=1e-3, seed=42):
    torch.manual_seed(seed)
    model = SeqClassifier(len(itos), emb_init=emb_init, cell=cell, pool=pool).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    y = torch.tensor(ytr).to(DEVICE)

    for ep in range(epochs):
        model.train()
        perm = torch.randperm(len(ytr))
        for i in range(0, len(ytr), bs):
            idx = perm[i:i + bs]
            opt.zero_grad()
            loss = loss_fn(model(Xtr[idx].to(DEVICE), ltr[idx]), y[idx])
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step()
        print(f"  epoch {ep + 1}: test accuracy {evaluate(model, Xte, lte, yte):.4f}")
    return model

Run the RNN. On two CPU threads this takes roughly 100 seconds.

In [ ]:
rnn_last = train(cell="rnn", pool="last")

---

## Part 4 — The LSTM

Session 25 built the LSTM out of two vectors and three gates, and put a parameter count against
it: four times the RNN's, for the four weight matrices. In PyTorch the entire change is one word.

    nn.RNN(...)   ->   nn.LSTM(...)

The only other difference visible from outside is what the layer returns. `nn.RNN` gives back
<code>h_n</code>; `nn.LSTM` gives back the pair <code>(h_n, c_n)</code> — session 25's two
vectors, exactly, which is why `SeqClassifier.forward` has that one branch in it.

In [ ]:
lstm_last = train(cell="lstm", pool="last")

### Reading the comparison

| cell | final hidden state, last 300 tokens |
|---|---|
| `nn.RNN` | 0.580 |
| `nn.LSTM` | 0.776 |

Nearly twenty points, from one word of difference in the constructor. This is session 24's
argument as a measurement rather than a claim: the final hidden state of a 300-step sequence is
the end of a 300-link gradient chain, and in a plain recurrence the early links receive almost
nothing. The LSTM's cell state is the mechanism session 25 introduced for exactly this, and it is
the difference between a model that works and one barely above chance.

Hold onto the size of that gap. Part 5 makes it disappear.

---

## Part 5 — Pooling, and a fair comparison

Taking only the final hidden state throws away 299 vectors the model already computed. Session 23
noted that pooling the hidden states is legitimate in a way that averaging raw word vectors is
not: every <code>h_i</code> already encodes everything before position <code>i</code>, so an
average of hidden states is not order-blind the way lab 8's document vectors were.

Our `SeqClassifier` already supports it — `pool="mean"`, averaging over the real timesteps only,
which is what the mask in `forward` is for.

In [ ]:
lstm_mean = train(cell="lstm", pool="mean")

Now the same change to the RNN, which is the comparison that matters.

In [ ]:
rnn_mean = train(cell="rnn", pool="mean")

Mean-pooling helps, and it helps for a reason worth naming: it gives every timestep a direct path
to the loss, so a gradient reaching step 5 no longer has to survive 295 steps of recurrence. That
is the same vanishing-gradient problem from session 24, dodged architecturally rather than solved.

Now run the RNN the same way, and put the four numbers side by side:

| cell | final state | mean-pooled |
|---|---|---|
| `nn.RNN` | 0.580 | 0.766 |
| `nn.LSTM` | 0.776 | 0.760 |

**The LSTM's advantage is gone.** Not reduced — gone, to within the couple of points this task
moves between epochs anyway. The gates were never valuable in themselves; they were valuable
because the final-state architecture asked the network to carry information across 300 steps.
Change the architecture so nothing has to be carried that far, and a plain RNN does the same work.

This is worth more than either number. An architectural fix and a pooling trick solved the same
problem, and the measurement says so plainly.

---

## Part 6 — Against tf-idf

Lab 8 left a table of five results on this exact task. We can now add rows to it. The comparison
holds the classifier's job fixed — every row predicts the same 500 labels from the same 1,500
training documents.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

tfidf = TfidfVectorizer(sublinear_tf=True)
A = tfidf.fit_transform(Xtr_text)
B = tfidf.transform(Xte_text)
tfidf_acc = LogisticRegression(max_iter=2000).fit(A, ytr).score(B, yte)
print("tf-idf + logistic regression:", round(tfidf_acc, 4))

# Verified output:
#   tf-idf + logistic regression: 0.844
# which is lab 8's own number on this split, unchanged.

### Reproducing the table

The table below is not something this notebook can produce in a lab slot: three splits times
three cells is nine training runs, about fifteen minutes on CPU. The code is here so the number
is checkable rather than asserted, and so anyone curious can leave it running afterwards.

In [ ]:
# Optional, ~15 minutes on CPU. Retrains every cell on every split.
def paired_trial(cells=("rnn", "lstm", "gru"), n_splits=3, epochs=8):
    """Score each cell on the same n_splits reshuffled splits. Returns {cell: [accuracies]}."""
    # YOUR CODE HERE
    # For each seed in range(n_splits):
    #   1. train_test_split(texts, labels, test_size=500, random_state=seed, stratify=labels)
    #   2. rebuild the vocabulary from THIS split's training documents, and re-encode both halves
    #   3. rebuild emb_init against the new vocabulary
    #   4. train each cell with pool="mean" and record its test accuracy
    # Rebuilding the vocabulary per split is the part that is easy to skip and wrong to skip:
    # a vocabulary built from documents that are now in the test set leaks.
    pass

### The result

One split is one split, and lab 7 spent an hour establishing that a difference this size needs
more than that. Running all three cells over three reshuffled splits — the same paired protocol,
each model retrained from scratch on each split — gives:

| representation | mean over 3 splits | std | the three splits |
|---|---|---|---|
| sublinear tf-idf | **0.865** | 0.005 | 0.870 · 0.864 · 0.860 |
| GRU, GloVe init, mean-pooled | 0.793 | 0.011 | 0.798 · 0.780 · 0.800 |
| RNN, GloVe init, mean-pooled | 0.789 | 0.020 | 0.790 · 0.768 · 0.808 |
| LSTM, GloVe init, mean-pooled | 0.773 | 0.008 | 0.782 · 0.770 · 0.768 |

**Tf-idf wins on every split, by seven to nine points.** The run-to-run spread is half a point to
two points, so a gap that size is not something a seed produced.

Two things in that table are worth reading slowly. The first is that the three recurrent cells are
separated by less than their own standard deviations — whatever ranks first here is ranked by
noise, and the LSTM, the most elaborate of the three, ranks last. The second is that all three
lose to a method from the 1970s that does not know what order the words came in.

This is the third lab in a row where the newer, more sophisticated representation loses to the
older one, and the reason is the same each time and is not the architecture. A recurrent network
has to learn what a word means, how words combine, and what sentiment is, from 1,500 documents.
Tf-idf has to learn a weight per word from the same 1,500 documents, and it has far fewer things
to be wrong about. Session 32's answer to this is pretraining: learn the language part from
billions of words elsewhere, and arrive at the 1,500 documents already knowing how English works.

We can see the shape of that answer in this lab already, in miniature. Starting the embedding
layer from GloVe is the same move at the smallest possible scale — one layer, initialised from
somebody else's corpus — and it was worth about four points. Session 32 pretrains *every* layer on
billions of words, and the difference between four points and closing a nine-point gap is mostly a
difference of scale.

---

## Part 7 — The GRU, in one word

Session 26 promised this would be a one-word change, and left the question of whether it matters
to a measurement rather than an argument. Make the change and see.

In [ ]:
# YOUR CODE HERE
# Train a GRU with mean pooling, exactly as the LSTM above was trained.

### What PyTorch actually computes

Session 26 wrote out Cho's equations and then flagged that `nn.GRU` is not equation (8) as
published: PyTorch applies the reset gate *after* the weight matrix rather than before. Its own
documentation says so. Rao & McMahan's chapter 7 — the source this lab follows — calls `nn.GRU`
without mentioning it.

That is worth holding onto beyond this lab. The equations in a paper and the layer in a library
are two different objects, and the second one is what your results come from.

---

## What this lab established

1. A document becomes a padded tensor plus a length, and the length is not bookkeeping — drop it
   and the model reads padding as content.
2. `nn.RNN`, `nn.LSTM` and `nn.GRU` are interchangeable in one word, differing only in what they
   return and what they cost.
3. Which 300 words you keep matters more than which of the three cells you choose.
4. On 1,500 documents, tf-idf beats all three, and the reason is data rather than architecture.

Lab 10 keeps the recurrent encoder and gives it something bag-of-words cannot do at all —
producing a *sequence* as output rather than one label, with attention deciding where to look.